# 07_SHAP_Explainability.ipynb

**Atmos Insight — SHAP Model Explainability Notebook**

## Introduction

This notebook provides end-to-end Explainable AI (XAI) feature attribution analysis for the production **Current AQI Random Forest Regressor**. Leveraging SHAP (SHapley Additive exPlanations), it computes global feature importances, beeswarm distribution plots, local waterfall step-by-step contributions, and force plot push vectors.

## Load Required Libraries

In [1]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pathlib import Path

# Resolve Project Root and SHAP Artifacts Output Directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SHAP_ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts' / 'shap'
SHAP_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root Resolved: {PROJECT_ROOT}")
print(f"Output Artifacts Path: {SHAP_ARTIFACTS_DIR}")

Project Root Resolved: C:\Users\sweet\AQI
Output Artifacts Path: C:\Users\sweet\AQI\artifacts\shap


## Load Trained Random Forest Model

In [2]:
model_path = PROJECT_ROOT / 'models' / 'current' / 'model.pkl'
model = joblib.load(model_path)
print(f"Loaded Model: {model.__class__.__name__}")
print(f"Number of Trees in Forest: {len(model.estimators_)}")

Loaded Model: RandomForestRegressor
Number of Trees in Forest: 100


## Load Scaler

In [3]:
scaler_path = PROJECT_ROOT / 'models' / 'current' / 'scaler.pkl'
scaler = joblib.load(scaler_path)
scale_cols = ['pm2.5', 'co', 'O3', 'NO2', 'SO2']
print(f"Loaded Scaler: {scaler.__class__.__name__}")
print(f"Feature Means: {dict(zip(scale_cols, scaler.mean_))}")

Loaded Scaler: StandardScaler
Feature Means: {'pm2.5': np.float64(153.11477380426825), 'co': np.float64(13.545126621440907), 'O3': np.float64(19.471338867058414), 'NO2': np.float64(28.82790511151017), 'SO2': np.float64(6.039191665508281)}


## Load Feature Columns

In [4]:
features_path = PROJECT_ROOT / 'models' / 'current' / 'feature_columns.pkl'
features = joblib.load(features_path)
print(f"Feature Columns ({len(features)} total): {features}")

Feature Columns (14 total): ['pm2.5', 'pm10', 'O3', 'NO2', 'SO2', 'co', 'Month', 'DayOfWeek', 'IsWeekend', 'season_Autumn', 'season_Monsoon', 'season_Spring', 'season_Summer', 'season_Winter']


## Load Test Dataset

In [5]:
df_orig = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'nehru_nagar_with_aqi.csv')
df_aug = pd.read_csv(PROJECT_ROOT / 'data' / 'processed' / 'clean_air_augmented_records.csv')
df_combined = pd.concat([df_orig, df_aug], ignore_index=True)

df_scaled = df_combined.copy()
df_scaled[scale_cols] = scaler.transform(df_scaled[scale_cols])
X_sample = df_scaled[features].sample(n=min(200, len(df_scaled)), random_state=42)
print(f"Combined Dataset Records: {len(df_combined)}")
print(f"Background Evaluation Sample Size: {len(X_sample)}")

Combined Dataset Records: 4376
Background Evaluation Sample Size: 200


## Create SHAP TreeExplainer

In [6]:
t0 = time.time()
explainer = shap.TreeExplainer(model)
expected_val = float(np.ravel(explainer.expected_value)[0])
print(f"TreeExplainer initialized in {time.time() - t0:.4f}s.")
print(f"Expected Base AQI Value E[f(x)]: {expected_val:.2f}")

shap_values_sample = explainer(X_sample)

TreeExplainer initialized in 0.0110s.
Expected Base AQI Value E[f(x)]: 301.21


## Generate Global Feature Importance

In [7]:
fig = plt.figure(figsize=(10, 6))
shap.plots.bar(shap_values_sample, show=False)
plt.title("Global SHAP Feature Importance (Current AQI Model)")
plt.tight_layout()
bar_path = SHAP_ARTIFACTS_DIR / 'global_feature_importance.png'
plt.savefig(bar_path, bbox_inches='tight')
plt.close(fig)
print(f"Saved: {bar_path}")

Saved: C:\Users\sweet\AQI\artifacts\shap\global_feature_importance.png


## Generate SHAP Summary Plot

In [8]:
fig = plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_sample.values, X_sample, feature_names=features, show=False)
plt.title("SHAP Summary Beeswarm Plot")
plt.tight_layout()
sum_path = SHAP_ARTIFACTS_DIR / 'shap_summary_plot.png'
plt.savefig(sum_path, bbox_inches='tight')
plt.close(fig)
print(f"Saved: {sum_path}")

Saved: C:\Users\sweet\AQI\artifacts\shap\shap_summary_plot.png


## Explain One Live OpenWeather Prediction

In [9]:
live_raw = {
    'pm2.5': 11.01,
    'pm10': 12.01,
    'O3': 52.74,
    'NO2': 2.71,
    'SO2': 1.81,
    'co': 0.18944,
    'Month': 8,
    'DayOfWeek': 5,
    'IsWeekend': 1,
    'season_Autumn': 0,
    'season_Monsoon': 1,
    'season_Spring': 0,
    'season_Summer': 0,
    'season_Winter': 0
}

df_live = pd.DataFrame([live_raw])[features]
df_live_scaled = df_live.copy()
df_live_scaled[scale_cols] = scaler.transform(df_live_scaled[scale_cols])

live_shap = explainer(df_live_scaled[features])
pred_aqi = model.predict(df_live_scaled[features])[0]
print(f"Live Sample Model Predicted AQI: {pred_aqi:.2f}")

Live Sample Model Predicted AQI: 52.24


## Generate SHAP Waterfall Plot

In [10]:
fig = plt.figure(figsize=(10, 6))
shap.plots.waterfall(live_shap[0], show=False)
plt.title("SHAP Waterfall Plot for Live Sample")
plt.tight_layout()
water_path = SHAP_ARTIFACTS_DIR / 'shap_waterfall_plot.png'
plt.savefig(water_path, bbox_inches='tight')
plt.close(fig)
print(f"Saved: {water_path}")

Saved: C:\Users\sweet\AQI\artifacts\shap\shap_waterfall_plot.png


## Generate SHAP Force Plot

In [11]:
fig = plt.figure(figsize=(12, 4))
shap.plots.force(expected_val, live_shap.values[0], df_live_scaled.iloc[0], matplotlib=True, show=False)
plt.title("SHAP Force Plot for Live Sample")
plt.tight_layout()
force_path = SHAP_ARTIFACTS_DIR / 'shap_force_plot.png'
plt.savefig(force_path, bbox_inches='tight')
plt.close(fig)
print(f"Saved: {force_path}")

Saved: C:\Users\sweet\AQI\artifacts\shap\shap_force_plot.png


## Save All Figures to artifacts/shap/

All visual SHAP figures (`global_feature_importance.png`, `shap_summary_plot.png`, `shap_waterfall_plot.png`, `shap_force_plot.png`) are automatically saved to `artifacts/shap/` relative to the project root.

## Conclusion

The SHAP explainability analysis confirms that `pm2.5` and `co` are the dominant negative features driving clean air predictions downwards from the historical baseline, while photochemical `O3` acts as a secondary positive contributor. All artifacts are fully reproducible.